In [2]:
# 2. 필요한 모듈 임포트
import pandas as pd
import plotly.io as pio
from IPython.display import display
import plotly.graph_objects as go

from src.services.data_preparer import prepare_proposal_04_data
from src.services.proposal_views.proposal_04_view import create_figure_and_df

# --- 테스트 실행 ---

# 1. app.py의 역할 (1): 데이터 준비 함수를 호출하여 필요한 데이터를 미리 로드합니다.
print("Step 1: `data_preparer`를 통해 분석용 데이터와 순서 정보를 준비합니다...")
data_bundle = prepare_proposal_04_data() # 글로벌 필터는 모두 기본값('전체')으로 호출
analysis_df = data_bundle.get("analysis_df", pd.DataFrame())
order_map = data_bundle.get("order_map", {})
print(" -> 데이터 준비 완료!")


# 2. app.py의 역할 (2): 사용자가 Streamlit 위젯을 통해 필터를 선택했다고 가정합니다.
# 이 값들을 바꾸면서 여러 경우를 테스트해볼 수 있습니다.

# 2-1. app.py에 있을 차원 설정(Dimension Config) 정의
DIMENSION_CONFIG = {
    '부서별': {'type': 'hierarchical', 'top': 'DIVISION_NAME', 'sub': 'OFFICE_NAME'},
    '직무별': {'type': 'hierarchical', 'top': 'JOB_L1_NAME', 'sub': 'JOB_L2_NAME'},
    '직위직급별': {'type': 'flat', 'col': 'POSITION_NAME'},
    '성별': {'type': 'flat', 'col': 'GENDER'},
    '연령별': {'type': 'flat', 'col': 'AGE_BIN'},
    '경력연차별': {'type': 'flat', 'col': 'CAREER_BIN'},
    '연봉구간별': {'type': 'flat', 'col': 'SALARY_BIN'},
    '지역별': {'type': 'flat', 'col': 'REGION_CATEGORY'},
    '계약별': {'type': 'flat', 'col': 'CONT_CATEGORY'}
}

# 2-2. 사용자 선택 시뮬레이션
# ----- 테스트하고 싶은 값으로 변경 -----
selected_dimension_ui = '연봉구간별' # 예: '부서별', '직무별', '성별', '직위직급별'
drilldown_selection = '전체'   # '부서별' 선택 시 'Development Division' 등으로 변경 가능
# -----------------------------------

print(f"Step 2: 사용자가 '{selected_dimension_ui}' 차원을, '{drilldown_selection}' 그룹으로 보기를 선택했습니다.")


# 3. app.py의 역할 (3): view 함수에 준비된 모든 데이터와 필터 값을 전달하여 결과물 생성
print("Step 3: `view` 모듈을 호출하여 그래프와 요약 테이블을 생성합니다...")
if not analysis_df.empty:
    # `app.py`에서는 data_bundle을 전역적으로 관리하거나 필요시 다시 로드할 수 있습니다.
    # 여기서는 view 함수가 data_bundle을 직접 받지 않으므로, 이 부분을 view 함수 호출 인자에 포함하지 않습니다.
    fig, aggregate_df = create_figure_and_df(
        analysis_df=analysis_df, 
        dimension_ui_name=selected_dimension_ui, 
        drilldown_selection=drilldown_selection,
        dimension_config=DIMENSION_CONFIG,
        order_map=order_map
    )
    print(" -> 생성 완료!")
else:
    print(" -> 분석할 데이터가 없어 빈 결과물을 생성합니다.")
    fig, aggregate_df = go.Figure(), pd.DataFrame()


# --- 결과 확인 ---

# 4. ipynb에서 생성된 그래프를 확인합니다.
print("\n--- [결과 1] 생성된 Plotly 그래프 ---")
# pio.renderers.default = 'vscode' 
fig.show()

# 5. ipynb에서 생성된 요약 테이블(aggregate_df)을 확인합니다.
print(f"\n--- [결과 2] '{selected_dimension_ui}' 기준 생성된 요약 테이블 ---")
display(aggregate_df)

Step 1: `data_preparer`를 통해 분석용 데이터와 순서 정보를 준비합니다...
 -> 데이터 준비 완료!
Step 2: 사용자가 '연봉구간별' 차원을, '전체' 그룹으로 보기를 선택했습니다.
Step 3: `view` 모듈을 호출하여 그래프와 요약 테이블을 생성합니다...
 -> 생성 완료!

--- [결과 1] 생성된 Plotly 그래프 ---



--- [결과 2] '연봉구간별' 기준 생성된 요약 테이블 ---


SALARY_BIN,합계,"4,000만원 미만","4,000~5,999만원","6,000~7,999만원","8,000~9,999만원",1억원 이상
TENURE_GROUP,,,,,,
3년 이하,162,38,114,10,0,0
3년초과~7년이하,168,2,111,45,10,0
7년 초과,150,0,7,62,49,32
합계,480,40,232,117,59,32
